# The Vinyl Decade

### How Genres, Formats, and Label Power Reshaped Recorded Music, 2016-2026

**Course:** Big Data and Cloud Computing - Lab 2  
**Authors:** Q Angos, Jon Robin San Diego  
**Date:** November 16, 2026  
**Dataset:** Discogs data dumps, snapshots of 1 January 2020 and 1 October 2026

> **Status: code final; results run on full data on `<RUN DATE>`.**

## Contents

1. [Executive summary](#1.-Executive-summary)
2. [Problem and research questions](#2.-Problem-and-research-questions)
3. [Data](#3.-Data)
4. [Methodology](#4.-Methodology)
5. [Results](#5.-Results)
   - 5.1 RQ1 - Genre and style shifts
   - 5.2 RQ2 - Format mix and the vinyl boom
   - 5.3 RQ3 - Label power
   - 5.4 Snapshot comparison
   - 5.5 RQ4 - Communities
   - 5.6 Artist clustering
6. [Discussion](#6.-Discussion)
7. [Limitations](#7.-Limitations)
8. [Conclusion](#8.-Conclusion)
9. [Dashboard](#9.-Dashboard)
10. [References](#10.-References)
- [Appendix A - Project files](#Appendix-A---Project-files)
- [Appendix B - Function reference](#Appendix-B---Function-reference)

## 1. Executive summary

TODO - write last. One short paragraph per research question, each with its headline number:

- **RQ1 (genres and styles):** TODO
- **RQ2 (formats and reissues):** TODO
- **RQ3 (label power):** TODO
- **RQ4 (communities):** TODO

## 2. Problem and research questions

The past decade reorganized recorded music. Streaming came to dominate listening while vinyl and cassettes returned as a collector market, styles kept splintering, and major labels kept absorbing independents. Most accounts of these shifts rely on sales charts or streaming counts, which track hits and say little about the long tail: what actually got pressed, by whom, in which scenes, and through which labels.

We ask:

1. **RQ1.** How did genre and style composition shift from 2016 to 2026, and has music become more fragmented?
2. **RQ2.** How did the format mix change, and how much of the vinyl boom was reissues rather than new music?
3. **RQ3.** How concentrated is release activity within the three majors' label trees, which artists, producers, and writers dominate each label's output, and how did ownership change between snapshots?
4. **RQ4.** What communities of artists, producers, and labels emerge from the data, and do they match genre tags?

## 3. Data

### 3.1 Source

[Discogs](https://www.discogs.com/data/) publishes monthly dumps of its user-built music database as four gzipped XML files: releases, masters, labels, and artists. We use two full snapshots.

| Snapshot | Role | Compressed size |
|---|---|---|
| 1 October 2026 | Main analysis | 11.7 GB |
| 1 January 2020 | Pre-pandemic comparison | 7.2 GB |

### 3.2 Scope

Releases dated **2016-2026**, and the artists, labels, and masters they link to, taken from the October 2026 dump. The January 2020 dump is used only for the snapshot comparison (section 5.4).

### 3.3 Raw files and key fields

| File | One row is | Key fields used |
|---|---|---|
| releases | One physical or digital release | `_id`, `released`, `country`, `genres`, `styles`, `formats` (name, descriptions), `labels` (id, catalog number), `artists`, `extraartists` (credits with roles), `companies` (e.g. pressing plants), `master_id` |
| masters | A group of releases of the same music | `_id`, `main_release`, `year` |
| labels | A label or company | `id`, `name`, `parentLabel`, `sublabels` |
| artists | A person or group | `id`, `name`, `members`, `groups`, `aliases` |

The full nested schemas are saved in `schemas/*.json` and are passed to Spark at read time so that it does not have to infer them.

### 3.4 Normalized tables

The nested XML is flattened once into the Parquet tables below (built in [`explorers/01_ingest_and_parse.ipynb`](explorers/01_ingest_and_parse.ipynb) with `vinyl.tables`).

In [ ]:
%load_ext autoreload
%autoreload 2

import json
from functools import reduce
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

from vinyl import (
    config, ingest, tables,          # data
    trends, formats, labels,         # RQ1, RQ2, RQ3
    snapshots, networks, clustering, # comparison, RQ4
    viz, export,                     # figures, dashboard tables
)

spark = ingest.get_spark(app_name="vinyl-decade-report")


def mark_partial(ax):
    """Shade the partial last year (2026 holds January-September only)."""
    ax.axvspan(config.LAST_FULL_YEAR + 0.5, config.YEAR_MAX + 0.5, color="grey", alpha=0.15)
    return ax

In [ ]:
# The normalized tables and what each one holds.
pd.DataFrame({"table": list(tables.TABLES), "description": list(tables.TABLES.values())})

In [ ]:
# Load the tables of both snapshots from Parquet (fast; no XML is read here).
# The 2020 snapshot only has the tables the snapshot comparison needs.
main = {name: ingest.read_table(spark, config.MAIN_SNAPSHOT, name) for name in tables.snapshot_tables(config.MAIN_SNAPSHOT)}
base = {name: ingest.read_table(spark, config.BASE_SNAPSHOT, name) for name in tables.snapshot_tables(config.BASE_SNAPSHOT)}

In [ ]:
# Table sizes, for the data description. None: not built for that snapshot.
pd.DataFrame(
    {
        "table": name,
        "rows_2026": main[name].count() if name in main else None,
        "rows_2020": base[name].count() if name in base else None,
    }
    for name in tables.TABLES
)

In [ ]:
# Numbers for the Data finding: releases in scope, linked artists and labels, style coverage.
n_releases = main["releases"].count()
n_with_style = main["release_styles"].select("release_id").distinct().count()
data_rows = [
    ("releases_in_scope", n_releases, None, f"releases dated {config.YEAR_MIN}-{config.YEAR_MAX} in the October 2026 snapshot"),
    ("linked_artists", main["release_artists"].select("artist_id").distinct().count(), None, "distinct main-artist ids on those releases"),
    ("linked_labels", main["release_labels"].select("label_id").distinct().count(), None, "distinct label ids on those releases"),
    ("releases_with_style", n_with_style, n_with_style / n_releases, "releases with at least one style; share of releases in scope"),
]

# Every release in scope has a usable year, because the year filter is applied at ingest. If the
# splitter's stats are still on disk, show how many dump records the filter kept instead.
try:
    stage = ingest.read_stage_stats(ingest.stage_path(config.MAIN_SNAPSHOT, "releases"))
    data_rows.append(("dump_records_kept", stage["records_kept"], stage["records_kept"] / stage["records_seen"],
                      f"release records kept by the year pre-filter, of {stage['records_seen']:,} in the dump"))
except (OSError, ValueError, KeyError):
    data_rows.append(("year_filter", None, None, "year filter applied at ingest"))
data_summary = pd.DataFrame(data_rows, columns=["metric", "count", "share", "note"]).astype({"count": "Int64"})
export.export_table(data_summary, "data_summary")
data_summary

> **Finding (TODO).** State the number of releases in scope, the number of linked artists and labels, and the share of releases with at least one style.

## 4. Methodology

### 4.1 Pipeline

```
 gzipped XML (8 files)                                   explorers/01
        |  read once with saved schemas (vinyl.ingest)
        v
 releases_staged (nested Parquet, 2016-2026 only)
        |  flatten (vinyl.tables)
        v
 normalized Parquet tables  ------------------------------------------+
        |                                                             |
        +--> trends      style shares, entropy, co-occurrence   explorers/02  -> 5.1
        +--> formats     format mix, reissue share              explorers/03  -> 5.2
        +--> labels      label trees, major share, HHI          explorers/04  -> 5.3
        +--> snapshots   parent changes, cataloguing lag        explorers/05  -> 5.4
        +--> networks    edge lists, centrality, Leiden         explorers/06  -> 5.5
        +--> clustering  features, node2vec, UMAP, HDBSCAN      explorers/07  -> 5.6
```

The gzipped XML is read exactly once per file. Gzip cannot be split, so each file is parsed by a single task; reading with a saved schema avoids a second full pass for schema inference. Everything downstream reads Parquet.

### 4.2 How this report is organized

This notebook is the report. It holds no analysis logic of its own: every number comes from a documented function in the `vinyl/` package, and every design choice was first worked out in an explorer notebook.

| Report section | Explorer notebook | Module | What it does |
|---|---|---|---|
| 3. Data | [`01_ingest_and_parse`](explorers/01_ingest_and_parse.ipynb) | `vinyl.config`, `vinyl.ingest`, `vinyl.tables` | Reads the XML with saved schemas and writes normalized Parquet tables |
| 5.1 RQ1 | [`02_genres_and_styles`](explorers/02_genres_and_styles.ipynb) | `vinyl.trends` | Yearly style shares, Shannon entropy, style co-occurrence |
| 5.2 RQ2 | [`03_formats_and_reissues`](explorers/03_formats_and_reissues.ipynb) | `vinyl.formats` | Format shares, reissue definitions, reissue share |
| 5.3 RQ3 | [`04_labels_and_majors`](explorers/04_labels_and_majors.ipynb) | `vinyl.labels` | Label trees, major vs independent share, artist concentration |
| 5.4 Comparison | [`05_snapshot_comparison`](explorers/05_snapshot_comparison.ipynb) | `vinyl.snapshots` | Parent changes and cataloguing lag between snapshots |
| 5.5 RQ4 | [`06_networks_and_communities`](explorers/06_networks_and_communities.ipynb) | `vinyl.networks` | Edge lists, centrality, Leiden communities, agreement with genres |
| 5.6 Clustering | [`07_artist_clustering`](explorers/07_artist_clustering.ipynb) | `vinyl.clustering` | Artist features, node2vec, UMAP, HDBSCAN |
| All figures | - | `vinyl.viz` | Shared plotting helpers |

Appendix B prints the documentation of every function used.

### 4.3 Methods in brief

- **Trends (RQ1).** Share of each year's releases carrying each style; Shannon entropy of the yearly style distribution as the fragmentation measure; per-year style co-occurrence networks.
- **Formats (RQ2).** Share of releases per format by year. A format is shown on its own when it reaches 1% of a full year's releases; the others are grouped as Other. A release counts as a reissue when it carries one of the tags Reissue, Repress or Remastered, or when its master is dated at least one year before the release. The report uses the broadest definition (`is_reissue_any`: either condition) and reports the tag-only (`is_reissue`) and master-only (`is_reissue_master`) definitions next to it. Reissue share is computed by year and by genre; the share of vinyl growth explained by reissues compares 2016 with the last full year.
- **Label power (RQ3).** Label trees rebuilt from `parentLabel` links; major versus independent share by year; top-artist share and Herfindahl index of releases per artist for each label.
- **Snapshot comparison.** Labels whose parent differs between January 2020 and October 2026; share of 2016-2019 releases added only after January 2020, used to estimate how undercounted 2025-2026 are. The October 2026 dump covers January-September of 2026, so 2026 is scaled by a seasonal factor of 0.739 (`config.PARTIAL_YEAR_SHARE`), the mean share of a year's month-dated releases that fall in January-September in 2016-2025 (range 0.728-0.747). The undercount is also computed without this factor.
- **Networks (RQ4).** Artist collaboration, producer/engineer credit, and artist-label graphs; centrality measures; Leiden community detection.
- **Clustering.** Artist feature vectors (style TF-IDF, format mix, career span, label count) combined with node2vec graph embeddings, reduced with UMAP, clustered with HDBSCAN.

### Parameters

| Item | Value | Where it is set |
|---|---|---|
| Last full year | 2025; 2026 (January-September) is marked partial | `config.LAST_FULL_YEAR` |
| Style movers | Share change 2016 to 2025, styles with at least 500 releases in either year | `trends.top_movers` call in 5.1 |
| Prolific artists (robustness check) | An artist-year with at least 300 releases is dropped | `config.PROLIFIC_ARTIST_MIN_PER_YEAR` |
| Style co-occurrence | Edges kept at 50 releases per year | `trends.style_cooccurrence` call in 5.1 |
| Entropy sensitivity | 5 random samples per year, sample size of the smallest full year, seed 42 | `trends.entropy_sensitivity` call in 5.1 |
| Reissue definition | `is_reissue_any` (reissue tag or master at least 1 year older) | `REISSUE_FLAG` in 5.2 |
| Majors | Matched on root label id: Universal 38404, Sony 353657, Warner 2345 | `config.MAJOR_ROOT_IDS` |
| Placeholder parent | Label 212 is not followed as a parent | `config.PLACEHOLDER_LABEL_IDS` |
| Self-released class | Label 1818 and every label named "Not On Label ...", counted as their own class and left out of the artist-label graph | `config.SELF_RELEASED_LABEL_IDS`, `config.SELF_RELEASED_LABEL_PATTERN` |
| Placeholder artists | "Various" (194) and "Unknown Artist" (355) are dropped | `networks.PLACEHOLDER_ARTIST_IDS` |
| Concentration | Labels and label groups with at least 50 release-artist pairs | `min_releases=50` in 5.3 |
| Graph edge weights | At least 2 shared releases (collaboration), 3 (credit), 3 (artist-label); releases with more than 20 main artists skipped | `config.GRAPH_MIN_WEIGHT` |
| Leiden | Resolution 1.0, seed 42 | `LEIDEN_RESOLUTION`, `LEIDEN_SEED` in 5.5 |
| Artist clustering input | Artists with at least 5 releases; styles used by at least 20 of them | `clustering.artist_profile`, `clustering.style_tfidf` calls in 5.6 |
| node2vec | 64 dimensions, 10 walks × 30 steps per node, p = q = 1 | `clustering.node2vec_embeddings` call in 5.6 |
| UMAP | 30 neighbours; 2 components for the plot, 10 for clustering | `config.UMAP_N_NEIGHBORS` |
| HDBSCAN | Minimum cluster size 50 | `config.HDBSCAN_MIN_CLUSTER_SIZE` |
| 2026 seasonal factor | 0.739 | `config.PARTIAL_YEAR_SHARE` |

## 5. Results

### 5.1 RQ1 - Genre and style shifts

*How did genre and style composition shift from 2016 to 2026, and has music become more fragmented?*

Exploration and parameter choices: [`explorers/02_genres_and_styles.ipynb`](explorers/02_genres_and_styles.ipynb). Functions: `vinyl.trends`.

In [ ]:
genre_shares = (
    trends.tag_shares_by_year(main["release_genres"], tag_col="genre").toPandas().sort_values(["genre", "year"])
)
ax = viz.plot_lines(genre_shares, "year", "share", "genre",
                    title="Share of each year's releases tagged with each genre, 2016-2026 (2026 is partial: Jan-Sep)",
                    ylabel="share of releases", percent=True)
mark_partial(ax)
viz.save_figure(ax.figure, "rq1_genre_shares")
export.export_table(genre_shares, "rq1_genre_shares")
genre_shares[genre_shares.year.isin([config.YEAR_MIN, config.LAST_FULL_YEAR])].pivot(index="genre", columns="year", values="share")

In [ ]:
START_YEAR, END_YEAR = config.YEAR_MIN, config.LAST_FULL_YEAR   # 2026 is partial, so the last full year ends the comparison
style_shares = trends.tag_shares_by_year(main["release_styles"], tag_col="style").cache()
style_movers = trends.top_movers(style_shares, start_year=START_YEAR, end_year=END_YEAR, min_releases=500).toPandas()
export.export_table(style_movers.assign(start_year=START_YEAR, end_year=END_YEAR), "rq1_style_movers")
pd.concat([style_movers.head(15), style_movers.tail(15)])   # 15 largest gains, then 15 largest losses

In [ ]:
# Robustness: the same ranking without the releases of prolific artist-years (>= config.PROLIFIC_ARTIST_MIN_PER_YEAR in one year).
prolific = trends.prolific_artist_years(main["release_artists"]).cache()
styles_clean = trends.drop_prolific_releases(main["release_styles"], main["release_artists"], prolific)
movers_clean = trends.top_movers(
    trends.tag_shares_by_year(styles_clean, tag_col="style"),
    start_year=START_YEAR, end_year=END_YEAR, min_releases=500,
).toPandas()

change_main = style_movers.set_index("style")["change"]
change_clean = movers_clean.set_index("style")["change"]
spearman = change_main.corr(change_clean, method="spearman")   # aligned on style
print(f"Spearman correlation of the style change, all releases vs without prolific artists: {spearman:.3f} "
      f"({len(change_main.index.intersection(change_clean.index))} styles in both)")

shown = pd.unique(pd.concat([style_movers.head(15), style_movers.tail(15), movers_clean.head(15), movers_clean.tail(15)])["style"])
movers_no_prolific = pd.DataFrame({"style": shown})
movers_no_prolific["change_main"] = movers_no_prolific["style"].map(change_main)
movers_no_prolific["change_no_prolific"] = movers_no_prolific["style"].map(change_clean)
movers_no_prolific["shrunk_by"] = movers_no_prolific["change_main"].abs() - movers_no_prolific["change_no_prolific"].abs()
movers_no_prolific["spearman_all_styles"] = spearman
movers_no_prolific = movers_no_prolific.sort_values("change_no_prolific", ascending=False)
export.export_table(movers_no_prolific, "rq1_movers_no_prolific")
display(prolific.toPandas())                                              # the prolific artist-years themselves
movers_no_prolific.sort_values("shrunk_by", ascending=False).head(10)     # styles that lose most of their change

> **Finding (TODO).** Which genres and styles gained and lost share, with numbers.

In [ ]:
entropy = (
    trends.entropy_decomposition(main["release_styles"], tag_col="style", base_year=config.YEAR_MIN)
    .toPandas().sort_values("year")
)
ax = viz.plot_lines(entropy, "year", "evenness", title="Style evenness (entropy / maximum entropy) by year (2026 is partial: Jan-Sep)",
                    ylabel="evenness (0-1)")
mark_partial(ax)
viz.save_figure(ax.figure, "rq1_style_evenness")
export.export_table(entropy, "rq1_entropy")
entropy

In [ ]:
# Is the change new styles appearing, or evener use of the styles that already existed in the base year?
bits = entropy.melt(id_vars="year", value_vars=["entropy", "entropy_base_tags"], var_name="measure", value_name="bits")
ax = viz.plot_lines(bits, "year", "bits", "measure",
                    title=f"Style entropy: all styles vs styles already used in {config.YEAR_MIN} (2026 is partial: Jan-Sep)",
                    ylabel="Shannon entropy (bits)")
mark_partial(ax)
viz.save_figure(ax.figure, "rq1_style_entropy_decomposition")

In [ ]:
# Does evenness depend on how many releases a year has? Equal-size random samples per year (about 6-8 min on the full data).
with clustering.timed("entropy_sensitivity"):
    sensitivity = (
        trends.entropy_sensitivity(main["release_styles"], main["releases"], tag_col="style", n_draws=5, seed=42)
        .toPandas().sort_values("year")
    )
lines = sensitivity.melt(id_vars="year", value_vars=["evenness_full", "evenness_sub_mean"], var_name="series", value_name="evenness")
ax = viz.plot_lines(lines, "year", "evenness", "series",
                    title="Style evenness: all releases vs equal-size samples (min-max band; 2026 is partial: Jan-Sep)")
ax.fill_between(sensitivity["year"], sensitivity["evenness_sub_min"], sensitivity["evenness_sub_max"], alpha=0.25, color="grey")
mark_partial(ax)
viz.save_figure(ax.figure, "rq1_entropy_sensitivity")
export.export_table(sensitivity, "rq1_entropy_sensitivity")
sensitivity

In [ ]:
cooccurrence = trends.style_cooccurrence(main["release_styles"], min_count=50).cache()
style_network = trends.cooccurrence_graph_stats(cooccurrence)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
viz.plot_lines(style_network, "year", "density", title="Density of the style co-occurrence network (2026 is partial)", ax=axes[0])
viz.plot_lines(style_network, "year", "modularity", title="Louvain modularity of the style co-occurrence network (2026 is partial)", ax=axes[1])
for ax in axes:
    mark_partial(ax)
fig.tight_layout()
viz.save_figure(fig, "rq1_style_network")
export.export_table(style_network, "rq1_style_network")
style_network

In [ ]:
# The strongest style pairs in the last full year (Jaccard index of the two styles' release sets).
cooccurrence.where(F.col("year") == config.LAST_FULL_YEAR).orderBy(F.desc("jaccard")).limit(15).toPandas()

> **Finding (TODO).** Has music become more fragmented? Entropy/evenness in 2016 vs the end year, and what the co-occurrence networks add.

### 5.2 RQ2 - Format mix and the vinyl boom

*How did the format mix change, and how much of the vinyl boom was reissues rather than new music?*

Exploration and parameter choices: [`explorers/03_formats_and_reissues.ipynb`](explorers/03_formats_and_reissues.ipynb). Functions: `vinyl.formats`.

In [ ]:
format_shares = formats.format_shares_by_year(main["release_formats"]).toPandas()
main_format_names = formats.main_formats(format_shares, min_share=0.01)   # >= 1% of releases in at least one full year
grouped_format_shares = (
    formats.format_shares_by_year(formats.group_minor_formats(main["release_formats"], main_format_names))
    .toPandas().sort_values(["format_name", "year"])
)
ax = viz.plot_lines(grouped_format_shares, "year", "share", "format_name",
                    title="Share of releases by format, 2016-2026 (minor formats grouped as Other; 2026 is partial: Jan-Sep)",
                    ylabel="share of releases", percent=True)
mark_partial(ax)
viz.save_figure(ax.figure, "rq2_format_shares")
export.export_table(grouped_format_shares, "rq2_format_shares")
print("Main formats, largest mean share first:", main_format_names)
grouped_format_shares.pivot(index="year", columns="format_name", values="share").round(3)

> **Finding (TODO).** How vinyl, CD, cassette, and digital shares moved.

In [ ]:
flagged = formats.add_master_reissue_flag(main["release_formats"], main["releases"], main["masters"], min_gap_years=1)
REISSUE_FLAG = "is_reissue_any"   # the broadest definition (reissue tag or older master), reported alongside the other two
REISSUE_FLAGS = ["is_reissue", "is_reissue_master", REISSUE_FLAG]

vinyl_releases = flagged.where(F.col("format_name") == "Vinyl")
by_flag = reduce(
    lambda a, b: a.unionByName(b),
    [formats.reissue_share(vinyl_releases, by=["year"], flag_col=f).withColumn("flag", F.lit(f)) for f in REISSUE_FLAGS],
).cache()
vinyl_reissues = by_flag.toPandas().sort_values(["flag", "year"])

long = (vinyl_reissues[vinyl_reissues.flag == REISSUE_FLAG]
        .melt(id_vars="year", value_vars=["n_new", "n_reissues"], var_name="kind", value_name="n"))
ax = viz.plot_stacked_share(long, "year", "n", "kind",
                            title=f"Vinyl releases that are new or reissues ({REISSUE_FLAG}), 2016-2026 (2026 is partial: Jan-Sep)")
viz.save_figure(ax.figure, "rq2_vinyl_reissue_share")
export.export_table(vinyl_reissues, "rq2_vinyl_reissues")
vinyl_reissues.pivot(index="year", columns="flag", values="reissue_share").round(3)

In [ ]:
# Share of the growth in vinyl releases (first year to last full year) that is reissues, for each definition.
growth = (
    formats.growth_explained_by_reissues(by_flag, config.YEAR_MIN, config.LAST_FULL_YEAR)
    .toPandas().sort_values("flag").assign(start_year=config.YEAR_MIN, end_year=config.LAST_FULL_YEAR)
)
export.export_table(growth, "rq2_growth_explained")
growth

In [ ]:
by_genre = (
    formats.reissue_share(
        vinyl_releases.join(main["release_genres"].select("release_id", "genre"), "release_id"),
        by=["year", "genre"], flag_col=REISSUE_FLAG,
    ).toPandas().sort_values(["genre", "year"])
)
ax = viz.plot_lines(by_genre, "year", "reissue_share", "genre",
                    title="Reissue share of vinyl releases by genre, 2016-2026 (2026 is partial: Jan-Sep)",
                    ylabel="reissue share of vinyl", percent=True)
mark_partial(ax)
viz.save_figure(ax.figure, "rq2_vinyl_reissue_by_genre")
export.export_table(by_genre, "rq2_reissue_by_genre")
by_genre[by_genre.year == config.LAST_FULL_YEAR].sort_values("n_releases", ascending=False)

In [ ]:
# How old is the music on releases that have a master? Age = release year - master year (at least 1).
AGE_FORMATS = ["Vinyl", "CD"]
reissue_age = pd.concat(
    [formats.reissue_age_distribution(flagged, fmt).toPandas().assign(format_name=fmt) for fmt in AGE_FORMATS],
    ignore_index=True,
).sort_values(["format_name", "year", "age_bin_order"])
print("Distinct releases whose master is dated after the release (left out):",
      {fmt: formats.count_master_after_release(flagged, fmt) for fmt in AGE_FORMATS})

age_labels = formats.age_bin_labels()
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, fmt in zip(axes, AGE_FORMATS):
    part = reissue_age[reissue_age.format_name == fmt]
    part = part.assign(age_bin=pd.Categorical(part["age_bin"], categories=age_labels, ordered=True))
    viz.plot_stacked_share(part, "year", "n_releases", "age_bin",
                           title=f"{fmt}: age of the master in years (releases with a master; 2026 is partial)", ax=ax)
fig.tight_layout()
viz.save_figure(fig, "rq2_reissue_age")
export.export_table(reissue_age, "rq2_reissue_age")
reissue_age.groupby(["format_name", "year"])["median_age"].first().unstack("format_name")

In [ ]:
# Pressing plants: how many vinyl releases name a "Pressed By" company at all, and which companies carry the most.
PRESSING_ROLE = "Pressed By"
plant_coverage = formats.pressing_plant_coverage(main["release_companies"], main["release_formats"], "Vinyl", PRESSING_ROLE).toPandas()
top_plants = formats.top_pressing_plants(main["release_companies"], main["release_formats"], "Vinyl", PRESSING_ROLE, top_n=15).toPandas()

ax = viz.plot_lines(top_plants, "year", "share_of_format_year", "company_name",
                    title=f"Top 15 '{PRESSING_ROLE}' companies: share of vinyl releases naming a '{PRESSING_ROLE}' company (2026 is partial)",
                    percent=True)
mark_partial(ax)
viz.save_figure(ax.figure, "rq2_pressing_plants")

pressing_plants = (
    plant_coverage.merge(top_plants.drop(columns="n_year_with_role"), on="year", how="left")
    .sort_values(["year", "rank"])
)
export.export_table(pressing_plants, "rq2_pressing_plants")
display(plant_coverage.sort_values("year"))
top_plants.drop_duplicates("company_id").sort_values("rank")[["rank", "company_id", "company_name", "total_full_years"]]

> **Finding (TODO).** What share of vinyl releases, and of vinyl growth, were reissues; which genres drive it.

### 5.3 RQ3 - Label power

*How concentrated is release activity within the three majors' label trees, and which artists, producers, and writers dominate each label's output?*

Exploration and parameter choices: [`explorers/04_labels_and_majors.ipynb`](explorers/04_labels_and_majors.ipynb). Functions: `vinyl.labels`.

In [ ]:
# Resolve every label to its root once and reuse it everywhere below: the walk over 2.4 M labels is slow.
with clustering.timed("resolve_label_roots"):
    label_roots = labels.resolve_label_roots(main["labels"]).cache()
    label_roots.count()   # fills the cache
tagged_labels = labels.tag_major_labels(label_roots).cache()   # matches config.MAJOR_ROOT_IDS
tagged_labels.groupBy("major").agg(F.count("*").alias("n_labels")).toPandas()

In [ ]:
major_share = labels.major_share_by_year(main["release_labels"], tagged_labels).toPandas().sort_values(["major", "year"])
ax = viz.plot_lines(major_share, "year", "share", "major",
                    title="Share of each year's releases on labels of Universal, Sony, Warner and independents (2026 is partial: Jan-Sep)",
                    ylabel="share of releases", percent=True)
mark_partial(ax)
viz.save_figure(ax.figure, "rq3_major_share")
export.export_table(major_share, "rq3_major_share")
major_share[major_share.year.isin([config.YEAR_MIN, config.LAST_FULL_YEAR])]

> **Finding (TODO).** Share of releases inside each major's tree by year, and the trend.

In [ ]:
# Concentration inside single labels: median HHI and median share of the top artist, by major.
real_artists = main["release_artists"].where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))   # drop "Various", "Unknown Artist"
concentration = labels.artist_concentration(main["release_labels"], real_artists, min_releases=50)
(concentration.join(tagged_labels.select("label_id", "major"), "label_id")
    .groupBy("major")
    .agg(F.count("*").alias("n_labels"),
         F.expr("percentile_approx(hhi, 0.5)").alias("median_hhi"),
         F.expr("percentile_approx(top_artist_share, 0.5)").alias("median_top_artist_share"))
    .toPandas())

In [ ]:
# Concentration of each whole label group (a major's tree, or the independents), overall and by year.
group_overall = labels.group_concentration(
    main["release_labels"], real_artists, tagged_labels, group_col="major", min_releases=50
).toPandas().assign(scope="overall")
group_by_year = labels.group_concentration(
    main["release_labels"], real_artists, tagged_labels, group_col="major", min_releases=50, by_year=True
).toPandas().assign(scope="year")
group_conc = (
    pd.concat([group_overall, group_by_year], ignore_index=True)
    .astype({"year": "Int64"}).sort_values(["scope", "major", "year"])
)
export.export_table(group_conc, "rq3_group_concentration")

ax = viz.plot_lines(group_by_year.sort_values(["major", "year"]), "year", "hhi", "major",
                    title="HHI of artist shares inside each label group, 2016-2026 (2026 is partial: Jan-Sep)")
mark_partial(ax)
viz.save_figure(ax.figure, "rq3_group_hhi")
group_conc[(group_conc.scope == "overall") | group_conc.year.isin([config.YEAR_MIN, config.LAST_FULL_YEAR])]

In [ ]:
# How much parent information do the labels of each year's releases have? Self-released placeholders get their own class.
# Keep the 312,527 self-released ids as a DataFrame; passing them as a Python list is extremely slow.
self_released = labels.self_released_label_ids(main["labels"]).cache()
parent_cov = labels.parent_coverage(main["release_labels"], tagged_labels, self_released=self_released).toPandas()
export.export_table(parent_cov, "rq3_parent_coverage")
parent_cov

In [ ]:
# Releases with a writer credit at the release level and at the track level.
writer_levels = labels.credit_level_coverage(main["release_credits"], config.WRITING_ROLES).toPandas().sort_values(["year", "level"])
export.export_table(writer_levels, "rq3_writer_levels")
writer_levels.pivot(index="year", columns="level", values="n_releases")

In [ ]:
# Top 5 producers/engineers and writers for a fixed set of flagship labels (ids checked against the 2026 labels table).
FLAGSHIP_LABELS = {
    1866: "Columbia", 681: "Atlantic", 2311: "Interscope", 8377: "Island",
    4241: "Def Jam", 23528: "Warp Records", 77343: "Sub Pop", 26465: "Domino",
}
flagship_releases = main["release_labels"].where(F.col("label_id").isin(list(FLAGSHIP_LABELS)))   # filter first: far fewer joins
columns = ["label_id", "label_name", "rank", "artist_name", "n_releases", "share"]
top_producers = labels.top_contributors(flagship_releases, main["release_credits"], config.PRODUCTION_ROLES, top_n=5).toPandas()
top_writers = labels.top_contributors(flagship_releases, main["release_credits"], config.WRITING_ROLES, top_n=5).toPandas()
print("Producers and engineers")
display(top_producers.sort_values(["label_id", "rank"])[columns])
print("Writers")
display(top_writers.sort_values(["label_id", "rank"])[columns])

> **Finding (TODO).** How dominated labels are by their top artists (major vs independent), and who the dominant producers and writers are.

### 5.4 Snapshot comparison

*How did ownership change between January 2020 and October 2026, and how undercounted are the most recent years?*

Exploration and parameter choices: [`explorers/05_snapshot_comparison.ipynb`](explorers/05_snapshot_comparison.ipynb). Functions: `vinyl.snapshots`.

In [ ]:
# Cache the two labels tables: the root walks and parent chains below read them many times.
label_cols = ["label_id", "label_name", "parent_id", "parent_name"]
labels_old = base["labels"].select(*label_cols).cache()
labels_new = main["labels"].select(*label_cols).cache()

parent_changes = snapshots.label_parent_changes(labels_old, labels_new).cache()
parent_changes.groupBy("change").count().orderBy(F.desc("count")).toPandas()

In [ ]:
# Largest labels (by in-scope releases) whose parent changed.
volume = main["release_labels"].groupBy("label_id").agg(F.countDistinct("release_id").alias("n_releases"))
(parent_changes.where(F.col("change").isin("changed_parent", "gained_parent", "lost_parent"))
    .join(volume, "label_id").orderBy(F.desc("n_releases")).limit(20).toPandas())

In [ ]:
# Do the three major roots exist in both snapshots, with the same parent?
root_check = snapshots.major_root_check(labels_old, labels_new)
for r in root_check[~root_check.ok].itertuples():
    print(f"CHECK {r.major} (root {r.root_id}): in 2020={r.in_old}, in 2026={r.in_new}, "
          f"parent in 2020={r.parent_id_old}, parent in 2026={r.parent_id_new}")
root_check

In [ ]:
# Labels that moved into, out of, or between major trees (roots resolved in each snapshot separately).
with clustering.timed("major_moves"):
    moves = snapshots.major_moves(labels_old, labels_new, main["release_labels"]).cache()
    print(f"{moves.count():,} labels changed major between the snapshots")
    move_counts = (moves.groupBy("move", "major_old", "major_new")
                   .agg(F.count("*").alias("n_labels"), F.sum("n_releases").alias("n_releases"))
                   .orderBy(F.desc("n_labels")).toPandas())
    top_moves = moves.orderBy(F.desc("n_releases"), "label_id").limit(20).toPandas()
export.export_table(top_moves, "cmp_major_moves")
display(move_counts)
top_moves[["label_id", "label_name", "major_old", "major_new", "move", "n_releases"]]

In [ ]:
# Hand check of the three largest moves: the parent chain of each label in 2020 and in 2026.
pd.set_option("display.width", 200)
for row in top_moves.head(3).itertuples():
    print(f"=== {row.label_id} {row.label_name}: {row.major_old} -> {row.major_new} ({row.n_releases:,} releases)")
    print(snapshots.explain_move(row.label_id, labels_old, labels_new).drop(columns="label_id").to_string(index=False))

> **Finding (TODO).** Which notable labels moved into or out of a major's tree.

In [ ]:
# Cataloguing lag: of each release year's releases in October 2026, how many were already in the January 2020 snapshot?
LAG_LAST_YEAR = int(config.BASE_SNAPSHOT) - 1   # 2020 in the January 2020 snapshot is a stub of a few days
lag = snapshots.cataloguing_lag(base["releases"], main["releases"]).cache()
lag_pd = lag.toPandas().sort_values("year")
lag_by_age = lag_pd[lag_pd.year <= LAG_LAST_YEAR].assign(age=lambda d: int(config.BASE_SNAPSHOT) - d.year)

ax = viz.plot_lines(lag_by_age, "year", "completeness",
                    title="Share of each year's releases already catalogued by January 2020, release years 2016-2019",
                    ylabel="already catalogued by Jan 2020", percent=True)
viz.save_figure(ax.figure, "cmp_cataloguing_lag")
export.export_table(lag_by_age, "cmp_lag")
lag_by_age.sort_values("age")[["age", "year", "n_new", "n_already", "n_added_later", "completeness"]]

In [ ]:
# Estimated number of releases per year after the undercount, with and without the 2026 seasonal factor.
yearly = main["releases"].groupBy("year").agg(F.count("*").alias("n_releases"))
undercount = snapshots.estimate_undercount(lag, yearly).toPandas()                          # config.PARTIAL_YEAR_SHARE: 2026 -> 0.739
undercount_flat = snapshots.estimate_undercount(lag, yearly, partial_year_share={}).toPandas()
cmp_undercount = undercount[["year", "n_releases", "age", "completeness", "partial_year", "n_estimated"]].merge(
    undercount_flat[["year", "n_estimated"]].rename(columns={"n_estimated": "n_estimated_no_seasonal"}), on="year"
).sort_values("year")
cmp_undercount["seasonal_factor"] = cmp_undercount.n_estimated / cmp_undercount.n_estimated_no_seasonal

long = cmp_undercount.melt(id_vars="year", value_vars=["n_releases", "n_estimated"], var_name="series", value_name="n")
ax = viz.plot_lines(long, "year", "n", series="series", ylabel="releases",
                    title="Releases per year as catalogued (n_releases) and after the undercount estimate (n_estimated)")
mark_partial(ax)
viz.save_figure(ax.figure, "cmp_undercount")
export.export_table(cmp_undercount, "cmp_undercount")
cmp_undercount

In [ ]:
# Does the lag differ by format? Each release is grouped by its 2026 formats; release years up to LAG_LAST_YEAR only.
early = main["releases"].where(F.col("year") <= LAG_LAST_YEAR).select("release_id", "year").cache()
LAG_FORMATS = ("Vinyl", "CD", "Cassette", "File")
lag_fmt = (
    snapshots.cataloguing_lag_by(base["releases"], early, main["release_formats"].where(F.col("format_name").isin(*LAG_FORMATS)), "format_name")
    .toPandas().sort_values(["format_name", "year"]).assign(age=lambda d: int(config.BASE_SNAPSHOT) - d.year)
)
ax = viz.plot_lines(lag_fmt, "year", "completeness", series="format_name", percent=True,
                    title="Share already catalogued by January 2020, by format, release years 2016-2019",
                    ylabel="already catalogued by Jan 2020")
viz.save_figure(ax.figure, "cmp_lag_by_format")
export.export_table(lag_fmt, "cmp_lag_by_format")
lag_fmt.pivot(index=["age", "year"], columns="format_name", values="completeness").round(3)

In [ ]:
# The same for the 8 genres with the most releases in those years.
top_genres = [
    r.genre for r in (main["release_genres"].where(F.col("year") <= LAG_LAST_YEAR).groupBy("genre")
                      .agg(F.countDistinct("release_id").alias("n")).orderBy(F.desc("n")).limit(8).collect())
]
lag_gen = (
    snapshots.cataloguing_lag_by(base["releases"], early, main["release_genres"].where(F.col("genre").isin(top_genres)), "genre")
    .toPandas().sort_values(["genre", "year"]).assign(age=lambda d: int(config.BASE_SNAPSHOT) - d.year)
)
ax = viz.plot_lines(lag_gen, "year", "completeness", series="genre", percent=True,
                    title="Share already catalogued by January 2020, by genre (top 8), release years 2016-2019",
                    ylabel="already catalogued by Jan 2020")
viz.save_figure(ax.figure, "cmp_lag_by_genre")
export.export_table(lag_gen, "cmp_lag_by_genre")
lag_gen.pivot(index=["age", "year"], columns="genre", values="completeness").round(3)

> **Finding (TODO).** Share of 2016-2019 releases added after January 2020, and the implied undercount of 2025-2026.

### 5.5 RQ4 - Communities

*What communities of artists, producers, and labels emerge from the data, and do they match genre tags?*

Exploration and parameter choices: [`explorers/06_networks_and_communities.ipynb`](explorers/06_networks_and_communities.ipynb). Functions: `vinyl.networks`.

In [ ]:
# Three graphs. Minimum edge weights come from config.GRAPH_MIN_WEIGHT; self-released labels (a DataFrame, see 5.3)
# are left out of the artist-label graph, otherwise "Not On Label" becomes one giant hub.
w = config.GRAPH_MIN_WEIGHT
with clustering.timed("edge lists"):
    collab_edges = networks.collaboration_edges(main["release_artists"], min_weight=w["collaboration"], max_artists_per_release=20).cache()
    credit_edges = networks.credit_edges(main["release_artists"], main["release_credits"], roles=config.PRODUCTION_ROLES, min_weight=w["credit"]).cache()
    artist_label_edges = networks.artist_label_edges(
        main["release_artists"], main["release_labels"], min_weight=w["artist_label"], exclude_label_ids=self_released
    ).cache()
    edge_counts = {"collaboration": collab_edges.count(), "credit": credit_edges.count(), "artist-label": artist_label_edges.count()}
pd.DataFrame({"graph": k, "min_weight": w[k.replace("-", "_")], "edges": n} for k, n in edge_counts.items())

In [ ]:
# Collect the edge lists on the driver and build the graphs. The artist-label graph uses prefixed ids (a123 / l123).
with clustering.timed("graphs"):
    edge_tables = {
        "collaboration": collab_edges.toPandas(),
        "credit": credit_edges.toPandas(),
        "artist-label": networks.prefixed_bipartite(artist_label_edges).toPandas(),
    }
    graphs = {k: networks.to_igraph(v) for k, v in edge_tables.items()}
pd.DataFrame({"graph": k, "nodes": g.vcount(), "edges": g.ecount()} for k, g in graphs.items())

In [ ]:
# Names only for the nodes that are in the graphs, then PageRank (betweenness stays off: too slow on these graphs).
with clustering.timed("names and centrality"):
    names = {
        "collaboration": networks.node_names(main["artists"], ids=graphs["collaboration"].vs["name"]),
        "credit": networks.node_names(main["artists"], ids=graphs["credit"].vs["name"]),
        "artist-label": networks.node_names(main["artists"], main["labels"], prefixed=True, ids=graphs["artist-label"].vs["name"]),
    }
    centrality = {k: networks.centrality_table(g, betweenness=False) for k, g in graphs.items()}

top_pagerank = pd.concat(
    [
        centrality[k].nlargest(15, "pagerank").merge(names[k], on="node", how="left")
        .assign(graph=k, rank=lambda d: np.arange(1, len(d) + 1), node=lambda d: d["node"].astype(str))
        for k in graphs
    ],
    ignore_index=True,
)[["graph", "rank", "node", "name", "degree", "strength", "pagerank"]]
export.export_table(top_pagerank, "rq4_top_pagerank")
top_pagerank

In [ ]:
LEIDEN_RESOLUTION, LEIDEN_SEED = 1.0, 42
with clustering.timed("Leiden"):
    communities = {k: networks.leiden_communities(g, resolution=LEIDEN_RESOLUTION, seed=LEIDEN_SEED) for k, g in graphs.items()}
for k, c in communities.items():
    sizes = c["community"].value_counts()
    print(f"{k:14s} communities {len(sizes):>8,}  >=50 nodes {(sizes >= 50).sum():>5,}  largest {sizes.iloc[0]:>8,}  modularity {c.attrs['modularity']:.4f}")

In [ ]:
# Agreement between the communities and each artist's dominant genre and style. Dominant tags are collected only for
# artists that are in a graph. The credit graph also holds people who are never a main artist: they have no tag.
artist_ids_in_graphs = sorted(
    set(networks.graph_artist_ids(graphs["collaboration"]))
    | set(networks.graph_artist_ids(graphs["credit"]))
    | set(networks.graph_artist_ids(graphs["artist-label"], prefixed=True))
)
in_graphs = spark.createDataFrame([(i,) for i in artist_ids_in_graphs], "artist_id long")
graph_artists = main["release_artists"].join(in_graphs, "artist_id")
dominant = {
    "genre": networks.artist_dominant_tag(graph_artists, main["release_genres"], "genre").toPandas().rename(columns={"artist_id": "node"}),
    "style": networks.artist_dominant_tag(graph_artists, main["release_styles"], "style").toPandas().rename(columns={"artist_id": "node"}),
}
# Artist-label nodes are prefixed, so prefix the tag table too (label nodes then have no tag).
node_tags = {
    (k, tag): (t.assign(node="a" + t["node"].astype(str)) if k == "artist-label" else t)
    for k in graphs for tag, t in dominant.items()
}

rows = []
for k, g in graphs.items():
    n_artist_nodes = len(networks.graph_artist_ids(g, prefixed=(k == "artist-label")))
    for tag in ("genre", "style"):
        rows.append({"graph": k, "tag": tag, "n_artist_nodes": n_artist_nodes,
                     **networks.community_tag_agreement(communities[k], node_tags[(k, tag)], tag_col=tag)})
agreement = pd.DataFrame(rows)
agreement["share_tagged"] = agreement["n_nodes"] / agreement["n_artist_nodes"]

graph_stats = pd.DataFrame({
    "graph": list(graphs),
    "min_weight": [w[k.replace("-", "_")] for k in graphs],
    "nodes": [g.vcount() for g in graphs.values()],
    "edges": [g.ecount() for g in graphs.values()],
    "communities": [communities[k]["community"].nunique() for k in graphs],
    "communities_ge50": [int((communities[k]["community"].value_counts() >= 50).sum()) for k in graphs],
    "largest_community_share": [communities[k]["community"].value_counts(normalize=True).iloc[0] for k in graphs],
    "modularity": [communities[k].attrs["modularity"] for k in graphs],
})
graph_summary = graph_stats.merge(agreement, on="graph")
export.export_table(graph_summary, "rq4_graph_summary")
graph_summary[["graph", "min_weight", "nodes", "edges", "communities", "communities_ge50", "modularity",
               "tag", "n_nodes", "share_tagged", "nmi", "ari", "purity"]].round(4)

In [ ]:
# The 10 largest communities of the collaboration and artist-label graphs: key nodes, most common genres and styles.
def top_communities(graph_name: str) -> pd.DataFrame:
    """Describe the 10 largest communities of one graph with genre and style tags side by side."""
    parts = {
        tag: networks.describe_communities(communities[graph_name], centrality[graph_name], names[graph_name],
                                           node_tags=node_tags[(graph_name, tag)], top_n=5, min_size=50, tag_col=tag).head(10)
        for tag in ("genre", "style")
    }
    genre = parts["genre"].rename(columns={"n_tagged": "n_genre_tagged", "top_tags": "top_genres", "tag_purity": "genre_purity"})
    style = parts["style"].rename(columns={"n_tagged": "n_style_tagged", "top_tags": "top_styles", "tag_purity": "style_purity"})
    return genre.merge(style[["community", "n_style_tagged", "top_styles", "style_purity"]], on="community", how="left")


communities_collab = top_communities("collaboration")
communities_artist_label = top_communities("artist-label")
export.export_table(communities_collab, "rq4_communities_collab")
export.export_table(communities_artist_label, "rq4_communities_artist_label")
pd.set_option("display.max_colwidth", 120)
communities_collab

In [ ]:
communities_artist_label

In [ ]:
# How stable are the collaboration communities when the Leiden resolution changes? (The credit graph sweep takes about 7 min and is skipped here.)
with clustering.timed("collaboration resolution sweep"):
    resolution_sweep = networks.resolution_sweep(
        graphs["collaboration"], resolutions=(0.5, 1.0, 2.0), seed=LEIDEN_SEED, reference=LEIDEN_RESOLUTION
    ).assign(graph="collaboration")
export.export_table(resolution_sweep, "rq4_resolution_sweep")
resolution_sweep

> **Finding (TODO).** How many communities, what the largest ones are, and how closely they follow genre tags (NMI, ARI, purity).

### 5.6 Artist clustering

*Do artists group into recognizable types when described by what they release and who they work with?*

Exploration and parameter choices: [`explorers/07_artist_clustering.ipynb`](explorers/07_artist_clustering.ipynb). Functions: `vinyl.clustering`.

> **Not runnable yet.** `clustering.node2vec_embeddings` and `clustering.combine_features` are stubs.

In [ ]:
# Artist features (Spark): profile of artists with at least 5 releases, and style TF-IDF for those artists.
real_artists = main["release_artists"].where(~F.col("artist_id").isin(list(networks.PLACEHOLDER_ARTIST_IDS)))   # as in 5.3
with clustering.timed("artist_profile"):
    profile = clustering.artist_profile(real_artists, main["release_formats"], main["release_labels"], min_releases=5).cache()
    print(f"{profile.count():,} artists kept")
kept_artists = real_artists.join(profile.select("artist_id"), "artist_id", "left_semi")
tfidf = clustering.style_tfidf(kept_artists, main["release_styles"], min_artists=20)

In [ ]:
# node2vec on the collaboration graph of the kept artists. The slow steps are cached under $DISCOGS_DATA_DIR/derived/<snapshot>/
# (written by explorer 07 or by this cell) and reused when they match this run. REUSE_CACHE = False recomputes everything.
REUSE_CACHE = True
DERIVED = Path(config.DATA_DIR) / "derived" / config.MAIN_SNAPSHOT
assert not str(DERIVED).startswith("/mnt/data"), "derived files must not go under /mnt/data"
DERIVED.mkdir(parents=True, exist_ok=True)

kept_edges = networks.collaboration_edges(kept_artists, min_weight=config.GRAPH_MIN_WEIGHT["collaboration"]).toPandas()
graph_nodes = set(kept_edges["src"]) | set(kept_edges["dst"])
print(f"{len(kept_edges):,} edges, {len(graph_nodes):,} nodes")

n2v_file = DERIVED / "artist_n2v.parquet"
embeddings = None
if REUSE_CACHE and n2v_file.exists():
    embeddings = pd.read_parquet(n2v_file)
    if set(embeddings["node"]) != graph_nodes:
        print("saved embeddings do not match this graph; recomputing")
        embeddings = None
    else:
        print("loaded", n2v_file)
if embeddings is None:
    with clustering.timed("node2vec"):
        embeddings = clustering.node2vec_embeddings(kept_edges, dimensions=64, walk_length=30, num_walks=10, p=1.0, q=1.0)
    embeddings.to_parquet(n2v_file, engine="pyarrow", index=False)
    print("saved", n2v_file)
embeddings.shape

In [ ]:
# Combine the features and reduce them with UMAP (2 components for the plot, 10 for clustering).
profile_pd = profile.toPandas()
tfidf_pd = tfidf.toPandas()

files = {name: DERIVED / f"{name}.npy" for name in ("features", "artist_ids", "embedding_2d", "embedding_10d")}
umap_file = DERIVED / "umap_params.json"
# Files written by explorer 07 have no params file; it used the reduce_umap default of 30 neighbours.
saved_neighbors = json.loads(umap_file.read_text())["n_neighbors"] if umap_file.exists() else 30

cached = REUSE_CACHE and all(f.exists() for f in files.values())
if cached:
    artist_ids = pd.Series(np.load(files["artist_ids"]), name="artist_id")
    cached = set(artist_ids) == set(profile_pd["artist_id"])
    if not cached:
        print("saved features do not match the artist profile; recomputing")
    elif saved_neighbors != config.UMAP_N_NEIGHBORS:
        print(f"saved UMAP used n_neighbors={saved_neighbors}, config says {config.UMAP_N_NEIGHBORS}; recomputing")
        cached = False
if cached:
    features, embedding_2d, embedding_10d = (np.load(files[k]) for k in ("features", "embedding_2d", "embedding_10d"))
    print("loaded features and UMAP embeddings from", DERIVED)
else:
    with clustering.timed("combine_features"):
        features, artist_ids = clustering.combine_features(profile_pd, tfidf_pd, embeddings)
    with clustering.timed("UMAP 2d"):
        embedding_2d = clustering.reduce_umap(features, n_components=2, n_neighbors=config.UMAP_N_NEIGHBORS)
    with clustering.timed("UMAP 10d"):
        embedding_10d = clustering.reduce_umap(features, n_components=10, n_neighbors=config.UMAP_N_NEIGHBORS)
    for name, array in (("features", features), ("artist_ids", np.asarray(artist_ids)),
                        ("embedding_2d", embedding_2d), ("embedding_10d", embedding_10d)):
        np.save(files[name], array)
    umap_file.write_text(json.dumps({"n_neighbors": config.UMAP_N_NEIGHBORS}))
    print("saved", ", ".join(f.name for f in files.values()), "to", DERIVED)
features.shape, embedding_10d.shape

In [ ]:
# HDBSCAN on the 10-dimensional embedding.
with clustering.timed("HDBSCAN"):
    cluster_labels = clustering.cluster_hdbscan(embedding_10d, min_cluster_size=config.HDBSCAN_MIN_CLUSTER_SIZE)
n_clusters = len(set(cluster_labels) - {-1})
noise_share = float((cluster_labels == -1).mean())
print(f"{n_clusters} clusters; {noise_share:.1%} of {len(cluster_labels):,} artists are noise")

In [ ]:
fig = clustering.plot_clusters_2d(embedding_2d, cluster_labels, title="Artist clusters (UMAP, 2 components; grey = noise)")
print(viz.save_figure(fig, "rq4_clusters"))
fig

In [ ]:
# One row per cluster: size, top styles, format mix, career span and example artists (noise is its own row).
summary = clustering.describe_clusters(cluster_labels, artist_ids, profile_pd, tfidf_pd)
cluster_table = summary.assign(
    top_styles=summary["top_styles"].map(lambda pairs: "; ".join(f"{style} ({value:.3f})" for style, value in pairs)),
    examples=summary["examples"].map(lambda names: "; ".join(map(str, names))),
)
export.export_table(cluster_table, "rq4_clusters")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)
cluster_table.head(30)

In [ ]:
# Compare the clusters with (a) Leiden communities on the same collaboration edges and (b) each artist's dominant genre.
with clustering.timed("Leiden on the clustered artists"):
    kept_communities = networks.leiden_communities(networks.to_igraph(kept_edges), resolution=LEIDEN_RESOLUTION, seed=LEIDEN_SEED)
vs_leiden = clustering.compare_partitions(
    cluster_labels, kept_communities["community"].to_numpy(), artist_ids, kept_communities["node"]
)
dominant_genre = networks.artist_dominant_tag(kept_artists, main["release_genres"], "genre").select("artist_id", "genre").toPandas()
vs_genre = clustering.compare_partitions(
    cluster_labels, dominant_genre["genre"].to_numpy(), artist_ids, dominant_genre["artist_id"]
)
cluster_agreement = pd.DataFrame(
    [
        {"compared_with": "Leiden communities (collaboration graph)", **{k: v for k, v in vs_leiden.items() if k != "contingency"}},
        {"compared_with": "dominant genre", **{k: v for k, v in vs_genre.items() if k != "contingency"}},
    ]
).assign(n_clusters=n_clusters, noise_share=noise_share)
export.export_table(cluster_agreement, "rq4_cluster_agreement")
display(cluster_agreement)
display(vs_leiden["contingency"])   # 10 largest clusters (rows) by 10 largest Leiden communities (columns)
vs_genre["contingency"]             # 10 largest clusters (rows) by 10 largest genres (columns)

In [ ]:
# A 2-D sample for the dashboard: at most 20,000 artists, seed 42.
n_points = min(20_000, len(artist_ids))
pick = np.sort(np.random.default_rng(42).choice(len(artist_ids), size=n_points, replace=False))
artist_names = profile_pd.set_index("artist_id")["artist_name"]
cluster_points = pd.DataFrame({
    "x": embedding_2d[pick, 0],
    "y": embedding_2d[pick, 1],
    "cluster": cluster_labels[pick],
    "artist_name": artist_names.reindex(np.asarray(artist_ids)[pick]).to_numpy(),
})
export.export_table(cluster_points, "rq4_cluster_points")
cluster_points.head()

> **Finding (TODO).** Number of clusters, what defines each, the share of unclustered artists, and how clusters relate to the Leiden communities.

In [ ]:
# Every table written for the dashboard, with a one-line description. Files go to export.DASHBOARD_DATA_DIR.
manifest = {
    "data_summary": "Releases in scope, linked artists and labels, share of releases with a style, and how the year filter was applied.",
    "rq1_genre_shares": "Share of each year's releases tagged with each genre.",
    "rq1_style_movers": "Style share in 2016 and in the last full year, with the change; all styles with at least 500 releases.",
    "rq1_entropy": "Style entropy and evenness by year, with entropy over 2016 styles only and the share of new styles.",
    "rq1_entropy_sensitivity": "Style evenness on all releases and on equal-size random samples (mean, min, max) by year.",
    "rq1_style_network": "Style co-occurrence network by year: nodes, edges, density, components, modularity.",
    "rq1_movers_no_prolific": "Top style movers with and without prolific artist-years, with the Spearman correlation of the two rankings.",
    "rq2_format_shares": "Share of each year's releases by format; minor formats grouped as Other.",
    "rq2_vinyl_reissues": "Vinyl releases per year: new, reissues and reissue share, for the three reissue definitions.",
    "rq2_growth_explained": "Share of the growth in vinyl releases explained by reissues, for the three definitions.",
    "rq2_reissue_by_genre": "Reissue share of vinyl releases by year and genre.",
    "rq2_reissue_age": "Age of the master on vinyl and CD releases by year and age bin, with the median age.",
    "rq2_pressing_plants": "Share of vinyl releases naming a pressing plant, and the top 15 plants by year.",
    "rq3_major_share": "Share of each year's releases on labels of Universal, Sony, Warner and independents.",
    "rq3_group_concentration": "Artist concentration (HHI, top artist share) of each label group, overall and by year.",
    "rq3_parent_coverage": "Releases by year and the parent information of their labels, with a self-released class.",
    "rq3_writer_levels": "Releases with a writer credit per year, at release level and at track level.",
    "cmp_major_moves": "The 20 labels with most releases that moved into, out of or between major trees from 2020 to 2026.",
    "cmp_lag": "Releases of 2016-2019 already catalogued by January 2020, by release year and age.",
    "cmp_undercount": "Releases per year as catalogued and as estimated, with and without the 2026 seasonal factor.",
    "cmp_lag_by_format": "Cataloguing lag by format, release years 2016-2019.",
    "cmp_lag_by_genre": "Cataloguing lag for the 8 largest genres, release years 2016-2019.",
    "rq4_graph_summary": "Nodes, edges, Leiden communities, modularity and agreement with genre and style for each graph.",
    "rq4_communities_collab": "The 10 largest Leiden communities of the collaboration graph, with key artists and tags.",
    "rq4_communities_artist_label": "The 10 largest Leiden communities of the artist-label graph, with key nodes and tags.",
    "rq4_resolution_sweep": "Leiden on the collaboration graph at several resolutions: communities, modularity, NMI with resolution 1.0.",
    "rq4_top_pagerank": "The 15 nodes with the highest PageRank in each graph, with names.",
    "rq4_clusters": "One row per HDBSCAN artist cluster: size, top styles, format mix, career span, example artists.",
    "rq4_cluster_agreement": "Agreement of the artist clusters with Leiden communities and with dominant genre.",
    "rq4_cluster_points": "A sample of up to 20,000 artists with their 2-D UMAP position and cluster.",
}
print(export.export_manifest(manifest))

## 6. Discussion

TODO - tie the four answers together:

- Does fragmentation of styles (5.1) go together with the shift in formats (5.2)?
- Is the vinyl boom led by majors reissuing catalogue or by independents pressing new music (5.2 with 5.3)?
- Do the communities (5.5) cut across label ownership (5.3)?

## 7. Limitations

- **Contributor bias.** Discogs reflects what contributors catalogued, which skews toward physical media and collector scenes. Digital-only music is under-represented.
- **Two snapshots, net changes only.** The comparison shows that a label's parent changed between January 2020 and October 2026, not when, and misses changes that were reversed.
- **Cataloguing lag.** Recent years are still being filled in. TODO: quote the completeness estimate from 5.4 and state which years are affected.
- **Partial final year.** The October 2026 snapshot covers nine months of 2026.
- **Label trees depend on contributors.** "Independent" means "no link to a major's tree in Discogs". Missing parent links understate the majors.
- **Release dates are free text.** Releases without a usable year are excluded. TODO: quote how many.
- TODO: any others found during exploration.

## 8. Conclusion

TODO

## 9. Dashboard

TODO - link to the dashboard and list the views it contains. The figures saved under `figures/` by `viz.save_figure` and the aggregated tables above are its inputs.

## 10. References

- Discogs. *Discogs Data Dumps.* https://www.discogs.com/data/ (snapshots of 2020-01-01 and 2026-10-01).
- Traag, V. A., Waltman, L., and van Eck, N. J. (2019). From Louvain to Leiden: guaranteeing well-connected communities. *Scientific Reports*, 9, 5233.
- Grover, A., and Leskovec, J. (2016). node2vec: Scalable feature learning for networks. *KDD '16*.
- McInnes, L., Healy, J., and Melville, J. (2018). UMAP: Uniform Manifold Approximation and Projection for dimension reduction. arXiv:1802.03426.
- Campello, R. J. G. B., Moulavi, D., and Sander, J. (2013). Density-based clustering based on hierarchical density estimates. *PAKDD 2013*.
- TODO: sources for the industry context in section 2.

## Appendix A - Project files

```
lab2/
├── main.ipynb                          this report
├── explorers/                          working notebooks, one per report section
│   ├── 01_ingest_and_parse.ipynb         XML -> Parquet (run once, slow)
│   ├── 02_genres_and_styles.ipynb        RQ1
│   ├── 03_formats_and_reissues.ipynb     RQ2
│   ├── 04_labels_and_majors.ipynb        RQ3
│   ├── 05_snapshot_comparison.ipynb      2020 vs 2026
│   ├── 06_networks_and_communities.ipynb RQ4
│   ├── 07_artist_clustering.ipynb        RQ4
│   └── executed/                         the explorers run on the full data, with their outputs
├── vinyl/                              documented functions used by all notebooks
│   ├── config.py      paths, snapshots, year window, constants
│   ├── ingest.py      Spark session, schema loading, XML and Parquet I/O
│   ├── tables.py      nested XML rows -> normalized tables
│   ├── trends.py      style shares, entropy, co-occurrence
│   ├── formats.py     format mix, reissue share
│   ├── labels.py      label trees, major share, concentration
│   ├── snapshots.py   parent changes, cataloguing lag
│   ├── networks.py    edge lists, centrality, Leiden
│   ├── clustering.py  features, node2vec, UMAP, HDBSCAN
│   ├── export.py      CSV tables and manifest for the dashboard
│   └── viz.py         plotting helpers
├── schemas/                            saved Spark schemas of the four raw files
├── data/                               raw/ and parquet/ (or set DISCOGS_DATA_DIR)
├── figures/                            written by viz.save_figure
└── dashboard/                          the dashboard
    └── data/                           CSV tables and manifest.json, written by vinyl.export
```

**Run order.** `explorers/01_ingest_and_parse.ipynb` once, then this notebook. The other explorers are independent of each other.

## Appendix B - Function reference

Printed from the docstrings, so it always matches the code.

In [ ]:
import inspect

modules = [config, ingest, tables, trends, formats, labels, snapshots, networks, clustering, viz, export]
rows = []
for module in modules:
    for name, func in inspect.getmembers(module, inspect.isfunction):
        if func.__module__ == module.__name__ and not name.startswith("_"):
            summary = (inspect.getdoc(func) or "").split("\n")[0]
            rows.append({"module": module.__name__, "function": name, "summary": summary})

pd.set_option("display.max_colwidth", None)
pd.DataFrame(rows)

In [ ]:
# Full documentation of any one function:
help(trends.shannon_entropy_by_year)